# Обучение модели для сервиса: Random Forest, online shoppers intention

Разведочный анализ и сравнение моделей лежат в `onlineshoppers.ipynb`. Здесь только то, что нужно сервису:
1. загрузить данные (без дублей, как в EDA-ноутбуке);
2. собрать **один `Pipeline`**: препроцессинг (log1p, one-hot) и Random Forest;
3. подобрать гиперпараметры (то же пространство поиска, критерий ROC-AUC);
4. выбрать порог классификации по out-of-fold предсказаниям train;
5. один раз оценить на тесте;
6. сохранить **joblib-бандл**: `{"pipeline": ..., "metadata": {...}}` в `artifact/model.joblib`.

Общий код (список фичей, `build_pipeline`, `pick_threshold`) живёт в `src/shoppers/training.py`, чтобы ноутбук и скрипт `uv run python -m shoppers.training` не расходились.

In [1]:
import datetime as dt
import joblib, sklearn
import numpy as np, pandas as pd
from scipy.stats import randint
from sklearn.metrics import (average_precision_score, classification_report, f1_score, recall_score, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, train_test_split

from shoppers.training import (DEFAULT_DATA, DEFAULT_OUT, FEATURES, MODEL_VERSION, SEED, build_pipeline, load_data, pick_threshold)

## 1. Данные

In [2]:
X, y = load_data(DEFAULT_DATA)
print(X.shape, "доля покупок:", round(y.mean(), 4))
print("пропусков:", int(X.isna().sum().sum()))
X.head(3)

(12205, 17) доля покупок: 0.1563
пропусков: 0


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,0
1,0,0.0,0,0.0,2,64.0,0.0,0.1,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,0
2,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,0


Пропусков в данных нет, поэтому в схеме запроса все поля обязательные. `SimpleImputer` в пайплайне оставлен как страховка: если в обучающие данные когда-нибудь придут NaN, сервис не упадёт.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_train.shape, X_test.shape

((9764, 17), (2441, 17))

## 2. Pipeline: препроцессинг + модель одним объектом

In [4]:
pipe = build_pipeline()
pipe

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('log',
                                                  Pipeline(steps=[('imp',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('log1p',
                                                                   FunctionTransformer(feature_names_out='one-to-one',
                                                                                       func=<ufunc 'log1p'>))]),
                                                  ['Administrative',
                                                   'Administrative_Duration',
                                                   'Informational',
                                                   'Informational_Duration',
                                                   'ProductRelated',
                                                   'ProductRelated_Duration',
                                                   'PageValues']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Month', 'VisitorType',
                                                   'OperatingSystems',
                                                   'Browser', 'Region',
                                                   'TrafficType']),
                                                 ('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['BounceRates', 'ExitRates',
                                                   'SpecialDay', 'Weekend'])],
                                   verbose_feature_names_out=False)),
                ('model', RandomForestClassifier(n_jobs=-1, random_state=42))])

## 3. Подбор гиперпараметров (критерий ROC-AUC)

In [5]:
param_dist = {
    "model__n_estimators": randint(100, 250),
    "model__max_depth": randint(3, 20),
    "model__min_samples_leaf": randint(1, 20),
    "model__max_features": ["sqrt", 0.3, 0.5],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, cv=cv, scoring="roc_auc", random_state=SEED, n_jobs=1)
search.fit(X_train, y_train)
print("best CV ROC-AUC:", round(search.best_score_, 4))
search.best_params_

best CV ROC-AUC: 0.9325


{'model__max_depth': 11,
 'model__max_features': 0.5,
 'model__min_samples_leaf': 18,
 'model__n_estimators': 231}

## 4. Порог классификации

Дисбаланс классов (около 16% покупок): порог 0.5 пропускает много покупателей. Берём порог, максимизирующий F1 класса «покупка», по out-of-fold предсказаниям train. Тест для выбора порога не используется.

In [6]:
best = search.best_estimator_
oof = cross_val_predict(best, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
threshold = pick_threshold(y_train, oof)
threshold

0.393

## 5. Оценка на отложенном тесте (один раз)

In [7]:
best.fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= threshold).astype(int)
metrics = {
    "cv_roc_auc": round(float(search.best_score_), 4),
    "test_roc_auc": round(float(roc_auc_score(y_test, proba)), 4),
    "test_pr_auc": round(float(average_precision_score(y_test, proba)), 4),
    "test_f1": round(float(f1_score(y_test, pred)), 4),
    "test_recall": round(float(recall_score(y_test, pred)), 4),
}
print(metrics)
print(classification_report(y_test, pred, target_names=["нет покупки", "покупка"]))

{'cv_roc_auc': 0.9325, 'test_roc_auc': 0.9345, 'test_pr_auc': 0.7578, 'test_f1': 0.6958, 'test_recall': 0.7094}
              precision    recall  f1-score   support

 нет покупки       0.95      0.94      0.94      2059
     покупка       0.68      0.71      0.70       382

    accuracy                           0.90      2441
   macro avg       0.81      0.82      0.82      2441
weighted avg       0.90      0.90      0.90      2441



## 6. Сохраняем joblib-бандл с паспортом

In [8]:
best.set_params(model__n_jobs=1)   # онлайн-инференс одной строки: потоки только мешают
metadata = {
    "model_version": MODEL_VERSION,
    "features": FEATURES,               # ПОРЯДОК важен: сервис делает reindex(columns=features)
    "threshold": threshold,
    "target": "Revenue",
    "best_params": {k: (v.item() if isinstance(v, np.generic) else v) for k, v in search.best_params_.items()},
    "metrics": metrics,
    "n_train": int(len(X_train)),
    "trained_at": dt.datetime.now(dt.UTC).isoformat(timespec="seconds"),
    "sklearn_version": sklearn.__version__,
    "dataset": "online_shoppers_intention.csv (Kaggle, imakash3011), duplicates dropped",
}
DEFAULT_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": best, "metadata": metadata}, DEFAULT_OUT)
print("saved ->", DEFAULT_OUT, f"({DEFAULT_OUT.stat().st_size / 1e6:.1f} MB)")
metadata

saved -> /home/claude/shoppers-mlops/artifact/model.joblib (4.8 MB)


{'model_version': '1.0.0',
 'features': ['Administrative',
  'Administrative_Duration',
  'Informational',
  'Informational_Duration',
  'ProductRelated',
  'ProductRelated_Duration',
  'BounceRates',
  'ExitRates',
  'PageValues',
  'SpecialDay',
  'Month',
  'OperatingSystems',
  'Browser',
  'Region',
  'TrafficType',
  'VisitorType',
  'Weekend'],
 'threshold': 0.393,
 'target': 'Revenue',
 'best_params': {'model__max_depth': 11,
  'model__max_features': 0.5,
  'model__min_samples_leaf': 18,
  'model__n_estimators': 231},
 'metrics': {'cv_roc_auc': 0.9325,
  'test_roc_auc': 0.9345,
  'test_pr_auc': 0.7578,
  'test_f1': 0.6958,
  'test_recall': 0.7094},
 'n_train': 9764,
 'trained_at': '2026-11-06T20:12:05+00:00',
 'sklearn_version': '1.6.1',
 'dataset': 'online_shoppers_intention.csv (Kaggle, imakash3011), duplicates dropped'}

## 7. Проверка: бандл читается и даёт тот же ответ

In [9]:
bundle = joblib.load(DEFAULT_OUT)
p2 = bundle["pipeline"].predict_proba(X_test)[:, 1]
assert np.allclose(p2, proba), "перезагруженная модель отвечает иначе"
print("ok:", bundle["metadata"]["model_version"], "порог", bundle["metadata"]["threshold"])

ok: 1.0.0 порог 0.393
